In [5]:
df = pd.DataFrame(
    [
        (2, 103, "2024-01-02 10:00:00", "2024-01-02 11:00:00"),
        (1, 101, "2024-01-01 09:00:00", "2024-01-01 10:00:00"),
        (3, 105, "2024-01-03 09:35:00", "2024-01-03 10:10:00"),
        (1, 104, "2024-01-02 14:00:00", "2024-01-02 15:30:00"),
        (2, 101, "2024-01-01 09:00:00", "2024-01-01 10:00:00"),
        (1, 107, "2024-01-04 13:00:00", "2024-01-04 14:00:00"),
        (3, 102, "2024-01-01 12:05:00", "2024-01-01 13:00:00"),
        (2, 108, "2024-01-05 15:10:00", "2024-01-05 16:00:00"),
        (1, 102, "2024-01-01 12:00:00", "2024-01-01 13:00:00"),
        (3, 107, "2024-01-04 13:00:00", "2024-01-04 14:00:00"),
        (2, 105, "2024-01-03 09:30:00", "2024-01-03 10:15:00"),
        (1, 108, "2024-01-05 15:00:00", "2024-01-05 16:00:00"),
        (3, 103, "2024-01-02 10:00:00", "2024-01-02 10:45:00"),
        (2, 106, "2024-01-04 11:00:00", "2024-01-04 12:00:00"),
        (1, 105, "2024-01-03 09:30:00", "2024-01-03 10:15:00"),
    ],
    columns=["employee_id", "meeting_id", "join_time", "leave_time"]
).assign(
    join_time=lambda x: pd.to_datetime(x["join_time"]),
    leave_time=lambda x: pd.to_datetime(x["leave_time"])
)

In [ ]:
df['date'] = df['join_time'].dt.date

df.sort_values(by=['employee_id', 'date', 'join_time'], inplace=True)

df['day_start'] = pd.to_datetime(df['date'].astype(str) + " " + '09:00:00')
df['day_end'] = pd.to_datetime(df['date'].astype(str) + " " + '17:00:00')

In [22]:
df

,employee_id,meeting_id,join_time,leave_time,date,day_start,day_end
1,1,101,2024-01-01 09:00:00,2024-01-01 10:00:00,2024-01-01,2024-01-01 09:00:00,2024-01-01 17:00:00
8,1,102,2024-01-01 12:00:00,2024-01-01 13:00:00,2024-01-01,2024-01-01 09:00:00,2024-01-01 17:00:00
3,1,104,2024-01-02 14:00:00,2024-01-02 15:30:00,2024-01-02,2024-01-02 09:00:00,2024-01-02 17:00:00
14,1,105,2024-01-03 09:30:00,2024-01-03 10:15:00,2024-01-03,2024-01-03 09:00:00,2024-01-03 17:00:00
5,1,107,2024-01-04 13:00:00,2024-01-04 14:00:00,2024-01-04,2024-01-04 09:00:00,2024-01-04 17:00:00
11,1,108,2024-01-05 15:00:00,2024-01-05 16:00:00,2024-01-05,2024-01-05 09:00:00,2024-01-05 17:00:00
4,2,101,2024-01-01 09:00:00,2024-01-01 10:00:00,2024-01-01,2024-01-01 09:00:00,2024-01-01 17:00:00
0,2,103,2024-01-02 10:00:00,2024-01-02 11:00:00,2024-01-02,2024-01-02 09:00:00,2024-01-02 17:00:00
10,2,105,2024-01-03 09:30:00,2024-01-03 10:15:00,2024-01-03,2024-01-03 09:00:00,2024-01-03 17:00:00
13,2,106,2024-01-04 11:00:00,2024-01-04 12:00:00,2024-01-04,2024-01-04 09:00:00,2024-01-04 17:00:00


In [45]:
working_tracker = {}
for i in range(len(df.index)):
    row = df.loc[df.index[i]]
    employee = row['employee_id']
    date_str = row['date'].strftime('%Y-%m-%d')
    leave_time = row['leave_time']
    if (employee, date_str) not in working_tracker:
        working_time_from_begin = (row['join_time'] - row['day_start']).seconds / 60
        working_time_to_end = (row['day_end'] - row['leave_time']).seconds / 60
        # print('working_time_from_begin: ', working_time_from_begin)
        # print('working_time_to_end: ', working_time_to_end)
        working_tracker[(employee, date_str)] = [leave_time, working_time_from_begin, working_time_to_end] # last meeting, longest time so far, longest time to end
    else:
        working_time = (row['join_time'] - working_tracker[(employee, date_str)][0]).seconds / 60
        working_tracker[(employee, date_str)][2] = (row['day_end'] - row['leave_time']).seconds / 60
        working_tracker[(employee, date_str)][1] = max( working_time, working_tracker[(employee, date_str)][1])
    

In [46]:
working_tracker

{(1, '2024-01-01'): [Timestamp('2024-01-01 10:00:00'), 120.0, 240.0],
 (1, '2024-01-02'): [Timestamp('2024-01-02 15:30:00'), 300.0, 90.0],
 (1, '2024-01-03'): [Timestamp('2024-01-03 10:15:00'), 30.0, 405.0],
 (1, '2024-01-04'): [Timestamp('2024-01-04 14:00:00'), 240.0, 180.0],
 (1, '2024-01-05'): [Timestamp('2024-01-05 16:00:00'), 360.0, 60.0],
 (2, '2024-01-01'): [Timestamp('2024-01-01 10:00:00'), 0.0, 420.0],
 (2, '2024-01-02'): [Timestamp('2024-01-02 11:00:00'), 60.0, 360.0],
 (2, '2024-01-03'): [Timestamp('2024-01-03 10:15:00'), 30.0, 405.0],
 (2, '2024-01-04'): [Timestamp('2024-01-04 12:00:00'), 120.0, 300.0],
 (2, '2024-01-05'): [Timestamp('2024-01-05 16:00:00'), 370.0, 60.0],
 (3, '2024-01-01'): [Timestamp('2024-01-01 13:00:00'), 185.0, 240.0],
 (3, '2024-01-02'): [Timestamp('2024-01-02 10:45:00'), 60.0, 375.0],
 (3, '2024-01-03'): [Timestamp('2024-01-03 10:10:00'), 35.0, 410.0],
 (3, '2024-01-04'): [Timestamp('2024-01-04 14:00:00'), 240.0, 180.0]}

In [62]:
res = pd.DataFrame(columns=['employee_id','date','longest_work_duration_minutes'])

In [63]:
print(len(res.columns))

3


In [68]:
for i, key in enumerate(working_tracker):
    value = working_tracker[key]
    longest = max(value[1], value[2])
    res.loc[i] = [key[0], key[1], longest]
res['date'] = pd.to_datetime(res['date'])

In [80]:
def longest_working_hour(df):
    df['date'] = df['join_time'].dt.date

    df.sort_values(by=['employee_id', 'date', 'join_time'], inplace=True)
    
    df['day_start'] = pd.to_datetime(df['date'].astype(str) + " " + '00:00:00')
    df['day_end'] = pd.to_datetime(df['date'].astype(str) + " " + '23:59:59')
    working_tracker = {}
    for i in range(len(df.index)):
        row = df.loc[df.index[i]]
        employee = row['employee_id']
        date_str = row['date'].strftime('%Y-%m-%d')
        leave_time = row['leave_time']
        if (employee, date_str) not in working_tracker:
            working_time_from_begin = (row['join_time'] - row['day_start']).total_seconds() / 60
            working_time_to_end = (row['day_end'] - row['leave_time']).total_seconds() / 60
            
            working_tracker[(employee, date_str)] = [leave_time, working_time_from_begin, working_time_to_end] # last meeting, longest time so far, longest time to end
        else:
            working_time = (row['join_time'] - working_tracker[(employee, date_str)][0]).total_seconds() / 60            
            working_tracker[(employee, date_str)][1] = max(working_time, working_tracker[(employee, date_str)][1])
            
            if row['leave_time'] > working_tracker[(employee, date_str)][0]:
                working_tracker[(employee, date_str)][2] = (row['day_end'] - row['leave_time']).total_seconds() / 60
                working_tracker[(employee, date_str)][0] = row['leave_time']
    
    res = pd.DataFrame(columns=['employee_id','date','longest_work_duration_minutes'])
    for i, key in enumerate(working_tracker):
        value = working_tracker[key]
        longest = max(value[1], value[2])
        res.loc[i] = [key[0], key[1], longest]
    res['date'] = pd.to_datetime(res['date'])
    return res
    

In [81]:
res = longest_working_hour(df)

In [82]:
res

,employee_id,date,longest_work_duration_minutes
0,1,2024-01-01,659.983333
1,1,2024-01-02,840.000000
2,1,2024-01-03,824.983333
3,1,2024-01-04,780.000000
4,1,2024-01-05,900.000000
5,2,2024-01-01,839.983333
6,2,2024-01-02,779.983333
7,2,2024-01-03,824.983333
8,2,2024-01-04,719.983333
9,2,2024-01-05,910.000000


In [83]:
df = pd.DataFrame([
    (1, "2024-01-01"),
    (1, "2024-01-02"),
    (1, "2024-01-04"),

    (2, "2024-01-01"),
    (2, "2024-01-01"),
    (2, "2024-01-03"),

    (3, "2024-01-02"),
    (3, "2024-01-03"),
    (3, "2024-01-05"),

    (4, "2024-01-02"),

    (5, "2024-01-03"),
    (5, "2024-01-04"),

    (6, "2024-01-03"),
    (6, "2024-01-05"),
], columns=["user_id", "activity_date"])

df["activity_date"] = pd.to_datetime(df["activity_date"])

In [84]:
df

,user_id,activity_date
0,1,2024-01-01
1,1,2024-01-02
2,1,2024-01-04
3,2,2024-01-01
4,2,2024-01-03
5,3,2024-01-02
6,3,2024-01-03
7,3,2024-01-05
8,4,2024-01-02
9,5,2024-01-03
